# Notebook 03: Initial results with a logistic regression baseline
**CIND 820 Milestone 3 · Canadian Income Survey 2018–2022 · RQ2 (out-of-time testing)**

How to run: open in Google Colab, choose **Runtime → Run all**, and allow Google Drive access when asked.
The notebook reads `ProcessedData/household_table.csv`, which Notebook 01 creates. It does not rebuild that file.

**Draft code prepared with Claude (Anthropic) as study material. The decisions marked YOUR DECISION, the notes under each step and the final explanations are mine.**

Each code cell imports the libraries it needs at the top, so you can see which library each step uses.

Plan in one paragraph: fit on 2018, check on 2019 and choose the cut-off probability on 2019 only, then refit on 2018 + 2019 and test once each on 2020, 2021 and 2022. Two tracks run side by side, one unweighted and one weighted by FWEIGHT.

## Step 1. Set up, open Drive and load the household table

In [ ]:
from pathlib import Path
import platform
import sys

import pandas as pd
import sklearn

# Open Google Drive when running in Colab. Outside Colab this import fails and we simply carry on.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# Same project folder as Notebook 00. Change only if yours is different.
PROJECT_PATH = Path("/content/drive/MyDrive/CIND820_CapstoneProject")
DATA_FILE = PROJECT_PATH / "ProcessedData" / "household_table.csv"
OUTPUT_PATH = PROJECT_PATH / "M3_outputs"
CODE_PATH = PROJECT_PATH / "code"                   # put m3_helpers.py in this folder

if not PROJECT_PATH.exists():
    raise FileNotFoundError(f"Cannot find the project folder {PROJECT_PATH}. In Colab, run this cell again and click Allow "
                            "when Google Drive asks for access, then check the folder name.")
if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Cannot find {DATA_FILE}. Run Notebook 01 first.")
if not (CODE_PATH / "m3_helpers.py").is_file():
    raise FileNotFoundError(f"Cannot find m3_helpers.py in {CODE_PATH}. Upload it to that folder.")

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)      # exist_ok=True: no error if the folder is already there
sys.path.append(str(CODE_PATH))                     # lets Python import m3_helpers from that folder

hh = pd.read_csv(DATA_FILE)
print("Household table:", hh.shape)
print("Python", platform.python_version(), "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

**Purpose:**

**What the code does:**

**My observed result:**

**Report section:**

**What would fail if this step were removed:**

## Step 2. Decisions fixed before any model is fitted
These are the settings the report will describe. Write the reasons in `analysis_plan.md` **before** looking at 2020–2022 results.
Changing any of them after seeing test results must be reported as exploratory.

In [ ]:
PREDICTORS = ["income_quintile", "tenure", "has_children", "SEX", "edu_household", "IMMST", "MAJRI", "PROV", "HHSIZE"]
from m3_helpers import TARGET, WEIGHT     # TARGET = "food_insecure" (1 = any food insecurity), WEIGHT = "FWEIGHT" (the earner's weight)

FIT_YEAR = 2018              # first fit
VALID_YEAR = 2019            # check the model and choose the cut-off here
TEST_YEARS = [2020, 2021, 2022]

C_VALUE = 1.0                # regularization strength of the logistic regression (scikit-learn default, no tuning)

# YOUR DECISION: the number the cut-off is chosen to maximize on 2019. Write the reason in analysis_plan.md.
# Options in this notebook: "f1" (precision and recall count equally), "f2" (recall counts twice as much as precision),
# or "balanced_accuracy" (recall and specificity count equally; it picks the same cut-off as the ROC method called Youden's J). The draft report says missing a food-insecure household is
# the costlier error, which fits "f2" better than "f1". Placeholder default below is "f1" until you decide.
THRESHOLD_METRIC = "f1"

# Columns that must never be predictors, because they contain or come from the outcome
assert TARGET not in PREDICTORS and "FSCHHLDM" not in PREDICTORS and WEIGHT not in PREDICTORS
print("Predictors:", PREDICTORS)

**Purpose:**

**What the code does:**

**My observed result:**

**Report section:**

**What would fail if this step were removed:**

## Step 3. Check the table against Milestone 2, and look at the outcome by year

In [ ]:
import numpy as np
import pandas as pd

# Household counts from the Milestone 2 report (Table 1)
EXPECTED_COUNTS = {2018: 39481, 2019: 30712, 2020: 52235, 2021: 38015, 2022: 40728}

counts = hh.groupby("YEAR").size().to_dict()
print("Counts match Milestone 2:", counts == EXPECTED_COUNTS)
assert counts == EXPECTED_COUNTS, "Household counts differ from the report. Stop and find out why."
assert not hh.duplicated(["YEAR", "PUMFID"]).any(), "A household appears twice."
assert hh[TARGET].isin([0, 1]).all()

rows = []
for year, g in hh.groupby("YEAR"):
    rows.append({
        "YEAR": year,
        "households": len(g),
        "food_insecure_n": int(g[TARGET].sum()),
        "unweighted_pct": round(100 * g[TARGET].mean(), 2),
        "weighted_pct": round(100 * np.average(g[TARGET], weights=g[WEIGHT]), 2),
    })
outcome_by_year = pd.DataFrame(rows)
outcome_by_year

**Purpose:**

**What the code does:**

**My observed result:**

**Report section:**

**What would fail if this step were removed:**

## Step 4. Split by year, never at random
Rows are never shuffled. Earlier years fit the model and later years test it, so the test never helps build the model.

In [ ]:
fit_df = hh[hh["YEAR"] == FIT_YEAR].copy()
valid_df = hh[hh["YEAR"] == VALID_YEAR].copy()
dev_df = hh[hh["YEAR"].isin([FIT_YEAR, VALID_YEAR])].copy()      # development data = 2018 + 2019
test_dfs = {y: hh[hh["YEAR"] == y].copy() for y in TEST_YEARS}

assert dev_df["YEAR"].max() < min(TEST_YEARS), "Test years must come after the development years."
print("Fit:", len(fit_df), "| Validation:", len(valid_df), "| Development:", len(dev_df))
print("Test:", {y: len(d) for y, d in test_dfs.items()})

**Purpose:**

**What the code does:**

**My observed result:**

**Report section:**

**What would fail if this step were removed:**

## Step 5. Descriptive statistics, histograms and boxplots (development years only)
`adj_income` is household income divided by the square root of household size. `HHSIZE` is the household size code (7 means 7 or more).
These summaries are unweighted. Outliers are flagged, not removed. The mode of a dollar amount is rarely meaningful (almost every value is different), so read it for HHSIZE and treat the income mode with caution.

In [ ]:
import pandas as pd

for var in ["adj_income", "HHSIZE"]:
    rows = []
    for year, g in dev_df.groupby("YEAR"):
        s = g[var].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        rows.append({
            "YEAR": year, "n": len(s), "mean": s.mean(), "median": s.median(),
            "mode": s.mode().iloc[0], "variance": s.var(),
            "outside_1.5_IQR": int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()),
            "zero_or_negative": int((s <= 0).sum()),
        })
    print(var)
    display(pd.DataFrame(rows).round(2))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for col, year in enumerate([2018, 2019]):
    s = dev_df.loc[dev_df["YEAR"] == year, "adj_income"].dropna()
    low, high = s.quantile(0.01), s.quantile(0.99)
    axes[0, col].hist(s[(s >= low) & (s <= high)], bins=50, color="#376986", edgecolor="white")
    axes[0, col].set(title=f"{year} adjusted income (1st to 99th percentile)", xlabel="dollars", ylabel="households")
    axes[1, col].boxplot(s)           # the boxplot shows the full range, so the outliers are visible
    axes[1, col].set(title=f"{year} adjusted income, full range", ylabel="dollars")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "adj_income_hist_box.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for col, year in enumerate([2018, 2019]):
    s = dev_df.loc[dev_df["YEAR"] == year, "HHSIZE"].dropna()
    axes[col].hist(s, bins=np.arange(0.5, 8.5, 1), color="#376986", edgecolor="white")
    axes[col].set(title=f"{year} household size", xlabel="HHSIZE code", ylabel="households")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "hhsize_hist.png", dpi=150)
plt.show()

**Purpose:**

**What the code does:**

**My observed result:**

**Report section:**

**What would fail if this step were removed:**

## Step 6. Helper functions (in `m3_helpers.py`)
The functions live in a separate file, `m3_helpers.py`, so this notebook stays readable and the code can be reused in Milestone 4. Open that file to read them.
- `make_X` turns the nine coded predictors into text labels, so each is treated as a category (one-hot encoding), not as a number. Missing values become their own category, "Missing".
- `fit_model` puts the encoder and the logistic regression in one pipeline, so the encoder learns its categories only from the data we fit on. That is how the leakage rule is kept.
- `predict_prob` gives each household's predicted probability of food insecurity.
- `confusion_scores` and `all_scores` give the confusion matrix and the metrics at one cut-off.
- `best_threshold` tries cut-offs from 0.05 to 0.95 and keeps the best one for the chosen metric.

In [ ]:
from m3_helpers import make_X, fit_model, predict_prob, confusion_scores, all_scores, best_threshold
print("Helper functions loaded from", CODE_PATH / "m3_helpers.py")

**Purpose:**

**What the code does:**

**My observed result:**

**Report section:**

**What would fail if this step were removed:**

## Step 7. Fit on 2018, check on 2019, choose the cut-off on 2019
Two tracks. **Unweighted:** every household counts once, scores are plain counts. **Weighted:** households count by FWEIGHT, scores use FWEIGHT.
Weighted results are working estimates, not official household estimates.

In [ ]:
from datetime import datetime
import pandas as pd

TRACKS = {"unweighted": False, "weighted": True}
thresholds = {}
valid_probs = {}
rows = []

for track, use_w in TRACKS.items():
    model = fit_model(fit_df, use_w, PREDICTORS, C_VALUE)                  # 2018 only
    prob = predict_prob(model, valid_df, PREDICTORS)
    valid_probs[track] = prob
    w = valid_df[WEIGHT] if use_w else None
    thresholds[track] = best_threshold(valid_df[TARGET], prob, w, THRESHOLD_METRIC)
    for label, t in [("cut-off 0.50 (reference)", 0.5), (f"cut-off maximizing {THRESHOLD_METRIC}", thresholds[track])]:
        rows.append({"track": track, "rule": label, **all_scores(valid_df[TARGET], prob, t, w)})

validation_table = pd.DataFrame(rows)
cols = ["track", "rule", "threshold", "accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
display(validation_table[cols].round(3))
validation_table.to_csv(OUTPUT_PATH / "validation_2019.csv", index=False)

# Record the cut-offs BEFORE any test year is scored
with open(OUTPUT_PATH / "frozen_thresholds.txt", "w") as f:
    f.write(f"Written {datetime.now()} before 2020-2022 were scored\n")
    f.write(f"metric={THRESHOLD_METRIC} C={C_VALUE} thresholds={thresholds}\n")
print("Frozen cut-offs:", thresholds)

**Purpose:**

**What the code does:**

**My observed result:**

**Report section:**

**What would fail if this step were removed:**

### Step 7b. ROC curve on 2019 (the ROC way of looking at cut-offs)
The ROC curve shows recall against the false positive rate (1 - specificity) for every possible cut-off. It does not choose a cut-off by itself.
Youden's J picks the point farthest above the diagonal, where recall + specificity - 1 is largest. ROC-AUC (area under the curve) does not depend on any cut-off.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (track, use_w) in zip(axes, TRACKS.items()):
    w = valid_df[WEIGHT] if use_w else None
    fpr, tpr, cuts = roc_curve(valid_df[TARGET], valid_probs[track], sample_weight=w)
    j = tpr - fpr                                  # Youden's J = recall + specificity - 1
    k = j.argmax()
    ax.plot(fpr, tpr, label="ROC curve")
    ax.plot([0, 1], [0, 1], "--", color="grey", label="no skill")
    ax.scatter(fpr[k], tpr[k], color="red", zorder=3, label=f"Youden J cut-off = {cuts[k]:.2f}")
    chosen = confusion_scores(valid_df[TARGET], valid_probs[track], thresholds[track], w)
    ax.scatter(1 - chosen["specificity"], chosen["recall"], color="green", zorder=3,
               label=f"{THRESHOLD_METRIC} cut-off = {thresholds[track]:.2f}")
    ax.set(title=f"2019 ROC, {track}", xlabel="False positive rate (1 - specificity)", ylabel="Recall")
    ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "roc_2019.png", dpi=150)
plt.show()

## Step 8. Refit on 2018 + 2019 and test once on 2020, 2021 and 2022
The cut-offs from Step 7 are used unchanged. Nothing is tuned on these years.

In [ ]:
import numpy as np
import pandas as pd

final_models = {track: fit_model(dev_df, use_w, PREDICTORS, C_VALUE) for track, use_w in TRACKS.items()}

rows = []
test_matrices = {}
for track, use_w in TRACKS.items():
    for year, df in test_dfs.items():
        prob = predict_prob(final_models[track], df, PREDICTORS)
        w = df[WEIGHT] if use_w else None
        result = all_scores(df[TARGET], prob, thresholds[track], w)
        rows.append({"YEAR": year, "track": track, "n": len(df), **result})
        test_matrices[(track, year)] = np.array([[result["TN"], result["FP"]], [result["FN"], result["TP"]]])

test_table = pd.DataFrame(rows)
cols = ["YEAR", "track", "n", "threshold", "accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
display(test_table[cols].round(3))
test_table.to_csv(OUTPUT_PATH / "test_2020_2022.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(11, 6.5))
for r, track in enumerate(TRACKS):
    for c, year in enumerate(TEST_YEARS):
        m = test_matrices[(track, year)]
        ax = axes[r, c]
        ax.imshow(m, cmap="Blues")
        ax.set(title=f"{year}, {track}", xticks=[0, 1], yticks=[0, 1],
               xticklabels=["Secure", "Insecure"], yticklabels=["Secure", "Insecure"],
               xlabel="Predicted", ylabel="Observed")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{m[i, j]:,.0f}", ha="center", va="center",
                        color="white" if m[i, j] > m.max() / 2 else "black")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "confusion_matrices_test.png", dpi=150)
plt.show()

In [ ]:
# Check one number by hand: recall for the unweighted track in 2022 = TP / (TP + FN)
row = test_table[(test_table["YEAR"] == 2022) & (test_table["track"] == "unweighted")].iloc[0]
by_hand = row["TP"] / (row["TP"] + row["FN"])
print(f"2022 recall = {row['TP']:,.0f} / ({row['TP']:,.0f} + {row['FN']:,.0f}) = {by_hand:.4f}  (table says {row['recall']:.4f})")
assert abs(by_hand - row["recall"]) < 1e-9

**Purpose:**

**What the code does:**

**My observed result:**

**Report section:**

**What would fail if this step were removed:**

## Step 9. Categories that appear in a test year but not in 2018–2019
Milestone 2 noted that MAJRI "no income" has no households in the training years. The encoder gives such a category all zeros, so the model cannot learn an effect for it.

In [ ]:
for year, df in test_dfs.items():
    for c in PREDICTORS:
        new = sorted(int(v) for v in set(df[c].dropna().unique()) - set(dev_df[c].dropna().unique()))
        if new:
            n = int(df[c].isin(new).sum())
            print(f"{year}: {c} has codes {new} not seen in 2018-2019 ({n} households, {100 * n / len(df):.2f}%)")
print("Done. Nothing printed above means no unseen categories.")

**Purpose:**

**What the code does:**

**My observed result:**

**Report section:**

**What would fail if this step were removed:**

## Step 10. Does the overlap between HHSIZE and has_children matter? (development data only)
Milestone 2 found Cramér's V of 0.78 between these two predictors. Here the model is fitted on 2018 and scored on 2019 with and without each of them.
ROC-AUC and PR-AUC are used because they do not depend on a cut-off. No test year is used.

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

variants = {
    "all nine predictors": PREDICTORS,
    "without has_children": [c for c in PREDICTORS if c != "has_children"],
    "without HHSIZE": [c for c in PREDICTORS if c != "HHSIZE"],
    "without both": [c for c in PREDICTORS if c not in ("has_children", "HHSIZE")],
}
rows = []
for name, cols in variants.items():
    for track, use_w in TRACKS.items():
        model = fit_model(fit_df, use_w, cols, C_VALUE)
        prob = predict_prob(model, valid_df, cols)
        w = valid_df[WEIGHT] if use_w else None
        rows.append({"model": name, "track": track,
                     "roc_auc": roc_auc_score(valid_df[TARGET], prob, sample_weight=w),
                     "pr_auc": average_precision_score(valid_df[TARGET], prob, sample_weight=w)})
overlap_check = pd.DataFrame(rows)
overlap_check.round(4).to_csv(OUTPUT_PATH / "hhsize_has_children_check.csv", index=False)
overlap_check.round(4)

**Purpose:**

**What the code does:**

**My observed result:**

**Report section:**

**What would fail if this step were removed:**

## Step 11. Interpretation (write in your own words)
- Which metrics move between 2020, 2021 and 2022, and what the errors mean for households that really are food insecure
- How the weighted and unweighted tracks differ
- Why the cut-off chosen on 2019 may fit later years badly (the share of food-insecure households changes by year)
- Limits: working weights, income quintiles built from each year's own income distribution, one regularization setting, no tuning